# Assignment — Bloc 3a
### Mini-repte: El teu so processat

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

**Objectiu:** Grava (o carrega) un so, implementa com a mínim 2 efectes del catàleg i crea una versió processada final que combini almenys 2 efectes encadenats. Hi ha una part d'implementació (amb autotest) i una part d'exploració lliure.

No esborris les cel·les d'autotest.

In [ ]:
import numpy as np
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sample_rate = 44100

## Part 0 — El teu so d'entrada

**Opció A (recomanada):** grava la teva veu (3 segons).

In [ ]:
# Opció A — Gravar
from google.colab import output
from base64 import b64decode
from IPython.display import Javascript

RECORD_JS = """
const sleep = t => new Promise(r => setTimeout(r, t))
const b2text = blob => new Promise(r => {
  const reader = new FileReader()
  reader.onloadend = e => r(e.srcElement.result)
  reader.readAsDataURL(blob)
})
async function record(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(seconds * 1000)
  recorder.stop()
  await new Promise(r => recorder.onstop = r)
  stream.getTracks().forEach(t => t.stop())
  return await b2text(new Blob(chunks))
}
"""

display(Javascript(RECORD_JS))
print("Gravant 3 segons... fes un so ara!")
s = output.eval_js('record(3)')
b = b64decode(s.split(',')[1])
with open('gravacio_raw.webm', 'wb') as f:
    f.write(b)
!ffmpeg -y -i gravacio_raw.webm -ar 44100 -ac 1 input.wav -loglevel quiet
print("Gravat com input.wav")

In [ ]:
# Opció B — Usar fitxer d'exemple (si l'Opció A no funciona)
import urllib.request
base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/02_numpy_audio/sessio02_wav_io/recursos/audio/"
urllib.request.urlretrieve(base_url + "example_pad.wav", "input.wav")
print("Fitxer d'exemple descarregat com input.wav")

In [ ]:
# Carregar el so d'entrada (executa sempre aquesta cel·la)
data, sr = librosa.load('input.wav', sr=sample_rate)
print("Durada:", len(data)/sample_rate, "s")
Audio(data, rate=sample_rate)

## Part 1 — Implementa `echo`

Implementa la funció `echo`. Ha de retornar l'array original amb una còpia retardada `delay_seconds` segons i atenuada per `decay` sumada per sobre.

**Recorda:** usa `np.copy(data)` — no `result = data`.

In [ ]:
def echo(data, delay_seconds, decay=0.5, sample_rate=44100):
    delay_samples = int(delay_seconds * sample_rate)

    # TODO 1: crea 'result' com una COPIA de 'data' (np.copy)
    result = None  # <-- substitueix

    # TODO 2: suma a result[delay_samples:] el valor data[:-delay_samples] * decay
    # Pista: result[delay_samples:] += ...

    return result

In [ ]:
# AUTOTEST 1 — no modifiquis aquesta cel·la

test_data = np.zeros(1000)
test_data[0] = 1.0  # un impuls al principi

result = echo(test_data, delay_seconds=0.01, decay=0.5, sample_rate=44100)

assert result is not None, "La funció retorna None"
assert len(result) == 1000, f"Longitud incorrecta: {len(result)}"
assert np.isclose(result[0], 1.0), "L'impuls original hauria de conservar-se"
assert result[441] > 0.4, f"L'eco hauria d'aparèixer a ~441 mostres, valor: {result[441]:.3f}"

# Comprova que NO modifica l'array original (np.copy)
original_copy = np.copy(test_data)
_ = echo(test_data, delay_seconds=0.01, decay=0.5)
assert np.array_equal(test_data, original_copy), "La funció modifica l'array original! Usa np.copy()"

print("✅ Part 1 correcta!")
display(Audio(echo(data, delay_seconds=0.2, decay=0.5), rate=sample_rate))

## Part 2 — Implementa `distortion`

Implementa `distortion`: amplifica el senyal per `drive` i retalla els valors fora de `[-threshold, threshold]` amb `np.clip`. Normalitza el resultat perquè l'amplitud màxima sigui 1.

In [ ]:
def distortion(data, drive=5.0, threshold=0.7):
    # TODO 3: amplifica 'data' per 'drive' i retalla entre -threshold i threshold
    # Pista: np.clip(array, min, max)
    clipped = None  # <-- substitueix

    # TODO 4: normalitza perquè el valor màxim absolut sigui 1
    return None  # <-- substitueix

In [ ]:
# AUTOTEST 2 — no modifiquis aquesta cel·la

test_data = np.array([0.1, 0.5, 0.9, -0.5, -0.9])
result = distortion(test_data, drive=3.0, threshold=0.8)

assert result is not None, "La funció retorna None"
assert len(result) == 5, f"Longitud incorrecta: {len(result)}"
assert np.isclose(np.max(np.abs(result)), 1.0), \
    f"El màxim absolut hauria de ser 1.0, és {np.max(np.abs(result)):.3f}"
# Valors grans han de quedar retallats (iguals entre ells abans de normalitzar)
assert np.isclose(abs(result[2]), abs(result[4])), \
    "Els valors retallats haurien de tenir la mateixa amplitud absoluta"

print("✅ Part 2 correcta!")
display(Audio(distortion(data, drive=5.0), rate=sample_rate))

## Part 3 — Explora el catàleg i crea la teva versió final

Aquí tens la resta d'efectes ja implementats. **Prova'ls, canvia paràmetres, combina'ls**. L'objectiu és crear una versió final del teu so que sigui interessant per a tu.

In [ ]:
# Efectes proporcionats — no cal implementar-los, pots usar-los directament

def reverse(data):
    return data[::-1]

def fade_in(data, duration, sample_rate=44100):
    n = min(int(duration * sample_rate), len(data))
    env = np.ones(len(data))
    env[:n] = np.linspace(0, 1, n)
    return data * env

def fade_out(data, duration, sample_rate=44100):
    n = min(int(duration * sample_rate), len(data))
    env = np.ones(len(data))
    env[-n:] = np.linspace(1, 0, n)
    return data * env

def delay_multi(data, delay_seconds, decay=0.5, n_repeats=4, sample_rate=44100):
    result = np.copy(data)
    delay_samples = int(delay_seconds * sample_rate)
    for i in range(1, n_repeats + 1):
        start = i * delay_samples
        if start >= len(data): break
        result[start:] += data[:len(data)-start] * (decay ** i)
    return result

def tremolo(data, rate=5.0, depth=0.5, sample_rate=44100):
    t = np.linspace(0, len(data)/sample_rate, len(data), endpoint=False)
    lfo = 1 - depth * (0.5 + 0.5 * np.sin(2 * np.pi * rate * t))
    return data * lfo

def ring_modulation(data, carrier_freq=200.0, sample_rate=44100):
    t = np.linspace(0, len(data)/sample_rate, len(data), endpoint=False)
    return data * np.sin(2 * np.pi * carrier_freq * t)

def playback_speed(data, factor, sample_rate=44100):
    """Retorna (data, nou_sample_rate). Reprodueix: Audio(data, rate=nou_sr)"""
    return data, int(sample_rate * factor)

print("Efectes carregats. Ara explora!")

In [ ]:
# Espai d'exploració lliure — prova efectes i combinacions
# Pots afegir tantes cel·les com necessitis (+ a la barra d'eines)

# Exemple per començar:
prova = tremolo(data, rate=3.0, depth=0.6)
Audio(prova, rate=sample_rate)

In [ ]:
# TODO 5 — Crea la teva versió final
# Requisit: aplica ALMENYS 2 efectes encadenats sobre 'data'
# Dona-li un nom a la variable que reflecteixi el que has fet

versio_final = data  # <-- modifica aquesta línia i afegeix efectes

# Visualitza i escolta el resultat
librosa.display.waveshow(versio_final, sr=sample_rate)
plt.title("La meva versió final")
plt.show()

Audio(versio_final, rate=sample_rate)

In [ ]:
# Desa el resultat com a WAV
sf.write("versio_final.wav", versio_final, sample_rate)

# Descarrega'l al teu ordinador
from google.colab import files
files.download("versio_final.wav")
print("Fitxer descarregat!")

## Part 4 — Reflexió (resposta breu, en text)

Respon a les preguntes següents editant aquesta cel·la (fes doble clic per editar):

**1. Quins 2 efectes has usat a la versió final? En quin ordre i per quin motiu?**

*(escriu aquí)*

**2. Has provat `reverse`. Per quin motiu creus que és impossible aplicar-lo en temps real (mentre el so entra pel micròfon)?**

*(escriu aquí)*

**3. Canviar `sample_rate` a `Audio(data, rate=sr*2)` fa sonar el so més agut i més curt. Per qué? (Pensa en el que vam aprendre a la Sessió 1 sobre la relació entre `sample_rate` i freqüència.)**

*(escriu aquí)*

---
## 🚀 Challenges (opcional)

1. **Pitch shift honest**: implementa un pitch shift que canviï el to *sense* canviar la durada. Pista: necessitaràs `scipy.signal.resample` per reinterpretar les mostres — investiga com funciona.
2. **Effecte combinat creatiu**: crea una cadena de 4+ efectes que produeixi un resultat molt diferent del so original. Documenta els paràmetres que has triat i per quin motiu.
3. **Catàleg propi**: implementa un efecte nou que no és al catàleg (per exemple, `chorus`, `bitcrusher`, o `flanger` — busca com funcionen i intenta-ho).

In [ ]:
# El teu codi del Challenge aquí